# Alerts Pipeline — End-to-End QC

Runs all three stages (detect → facts → investigate) across every signal type, with inspection cells between each stage. Scoring is excluded.
| Cell | Stage | What happens |
|------|-------|--------------|
| 1 | Config | Set analysis date, optional store filter |
| 2 | Session | Connect to Snowflake, load config | | 3 | Data load — OSA | OSA + hierarchy joined, feedback cached | | 4 | Data load — WMD | Markdown log, sales, wastage, item/store hierarchy | | 5–7 | **Stage 1 — Detect** | Run all detectors, inspect anomalies by signal type | | 8–11 | **Stage 2 — Facts** | Enrich signals, inspect facts per signal type | | 12–14 | **Stage 3 — Investigate** | Generate narratives, inspect per signal type | | 15 | Full card preview | Read one complete alert card per signal type |

In [1]:
# ── Date window ────────────────────────────────────────────────────────────────
# Set ANALYSIS_DATE_STR to the last day of the week you want to analyse.
ANALYSIS_DATE_STR = '2026-03-20'

# ── Client config (None = base defaults; 'eastofengland' = client override) ────
CLIENT = None

# ── Optional: restrict to one store for a faster test run ─────────────────────
QC_STORE_ID = None   # e.g. 507 — set to None for all stores

# ── Fact-gathering parameters ─────────────────────────────────────────────────
PROJECTION_WEEKS = 4      # cost-if-not-fixed projection horizon
DRILLDOWN_TOP_N  = 3      # top-N PPGs in drilldown lists

import datetime

analysis_date   = datetime.date.fromisoformat(ANALYSIS_DATE_STR)
THIS_WEEK_END   = str(analysis_date)
THIS_WEEK_START = str(analysis_date - datetime.timedelta(days=6))
LAST_WEEK_END   = str(analysis_date - datetime.timedelta(days=7))
LAST_WEEK_START = str(analysis_date - datetime.timedelta(days=13))
LOOKBACK_START  = str(analysis_date - datetime.timedelta(days=69))  # ~10 weeks
WMD_LOOKBACK    = str(analysis_date - datetime.timedelta(days=27))  # 4 weeks for SKU detectors

print(f'Analysis date : {THIS_WEEK_END}')
print(f'This week     : {THIS_WEEK_START} – {THIS_WEEK_END}')
print(f'Last week     : {LAST_WEEK_START} – {LAST_WEEK_END}')
print(f'Lookback      : {LOOKBACK_START} – {THIS_WEEK_END}')
print(f'WMD window    : {WMD_LOOKBACK} – {THIS_WEEK_END}')
print(f'Client        : {CLIENT or "base defaults"}')
print(f'Store filter  : {QC_STORE_ID or "all stores"}')

Analysis date : 2026-03-20
This week     : 2026-03-14 – 2026-03-20
Last week     : 2026-03-07 – 2026-03-13
Lookback      : 2026-01-10 – 2026-03-20
WMD window    : 2026-02-21 – 2026-03-20
Client        : base defaults
Store filter  : all stores


In [2]:
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.pipeline.config.loader import load_config
from store_insight.pipeline.pipeline.session import create_session

cfg     = load_config(CLIENT)
print(f'Config loaded: client={cfg["client_name"]}')

Config loaded: client=default


In [3]:
session = create_session(cfg)

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/93a31d5d-1a5d-433f-baa0-138549cd2108/saml2?SAMLRequest=nZJRb5swFIX%2FCvKeARvSjlghFU2WLVrbRQld1b052CRWwGa%2BJrT79TMkkbqH9mEvCNnn3u%2F4nju5eakr7ygMSK1SRAKMPKEKzaXapegxX%2FgJ8sAyxVmllUjRqwB0M50Aq6uGZq3dq7X43QqwnmukgPYXKWqNopqBBKpYLYDagm6y%2BzsaBZgyAGGsw6FzCQfpWHtrGxqGXdcFXRxoswsjjHGIx6FT9ZJP6A2i%2BZjRGG11oatLyYt70zsIEuJRj3AKR1idC2%2BlOo3gI8r2JAL6Lc9X%2FurHJkdednndTCtoa2E2whxlIR7XdycD4BxATa5jPAragw%2FazTBgf1ojAlC6Kyt2EIWum9a61oH7C0vBw0rvpBvYcp6i5iB5%2Bf32Ycv07Pmoyi9ft%2BN1a4ie2adolhU212Kxj7fJ5mmd8ecCeT8v8UZ9vEuAVixVH6p1Rzi69vHIJ59zgimJKU4CkiS%2FkDd3oUrF7FB5cT74CGpZGA26tFpVUonB5ThmMeFX3CfMfUZxXPpbxrBP4uRqNC54RHAS9tFF6LQ%2BdDBipv81lEn4tsV5HR9cQsv5SleyePUW2tTMvh8gCchwIrlfDlIqaiarjHMjAFyQVaW7mRHMuq23phUonJ6o%2F%2B79

 pip install snowflake-connector-python[secure-local-storage]


## Data Load — OSA + Feedback

In [4]:
from snowflake.snowpark import functions as F

OSA_TABLE       = cfg['tables']['osa']
HIERARCHY_TABLE = cfg['tables']['hierarchy']
FEEDBACK_TABLE  = cfg['tables']['feedback']

print(f'OSA table      : {OSA_TABLE}')
print(f'Hierarchy table: {HIERARCHY_TABLE}')
print(f'Feedback table : {FEEDBACK_TABLE}')

osa_raw = (
    session.table(OSA_TABLE)
    .filter(F.col('ANALYSIS_DATE') >= LOOKBACK_START)
    .filter(F.col('ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('SITE_ID').alias('STORE_ID'),
        F.col('ITEM_ID').alias('ITEM_CODE'),
        F.col('ACTUAL_SALES_VOLUME'),
        F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'),
        F.col('ANALYSIS_DATE'),
        F.col('DEPARTMENT_ID'),
    )
)

item = (
    session.table(HIERARCHY_TABLE)
    .select(
        F.col('MASTER_ITEM_ID').alias('ITEM_CODE'),
        F.col('ITEM_ID'),
        F.col('DEPARTMENT_NAME'),
        F.col('SUBCLASS_NAME'),
        F.col('PRODUCT_PROFILE_GROUP_NAME'),
    )
    .distinct()
)

osa = (
    osa_raw
    .join(item, on=['ITEM_CODE'], how='left')
    .filter(F.col('DEPARTMENT_NAME').is_not_null())
    .cache_result()
)

feedback = (
    session.table(FEEDBACK_TABLE)
    .filter(F.col('ALERT_ANALYSIS_DATE') >= LAST_WEEK_START)
    .filter(F.col('ALERT_ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('SITE_ID').alias('STORE_ID'),
        F.col('SITE_NAME'),
        F.col('ALERT_TYPE_NAME'),
        F.col('ALERT_ANALYSIS_DATE'),
        F.col('INTERVENTION_ID'),
        F.col('TICK_AND_FLICK_FLAG'),
    )
    .cache_result()
)

print(f'\nOSA rows loaded & cached      : {osa.count():,}')
print(f'Feedback rows loaded & cached : {feedback.count():,}')

OSA table      : ANALYSIS.FINAL.LSM_ITEM_STORE_DAY_REPORTING_PERIOD
Hierarchy table: ANALYSIS.FINAL.LSM_ITEM_HIERARCHY
Feedback table : ANALYSIS.FINAL.LSM_ALERT_FEEDBACK

OSA rows loaded & cached      : 70,576,311
Feedback rows loaded & cached : 19,861


## Data Load — WMD (Markdown, Waste, Hierarchies)

In [ ]:
from snowflake.snowpark import Window
from snowflake.snowpark.types import LongType

MARKDOWN_LOG_TABLE   = cfg['tables'].get('markdown_log',        'ANALYSIS.FINAL.WMD_MARKDOWN_API')
MARKDOWN_SALES_TABLE = cfg['tables'].get('markdown_sales',      'ANALYSIS.FINAL.WMD_SALES')
WASTAGE_TABLE        = cfg['tables'].get('wastage',             'ANALYSIS.FINAL.WMD_WASTAGE')
WMD_ITEM_HIER_TABLE  = cfg['tables'].get('wmd_item_hierarchy',  'ANALYSIS.FINAL.WMD_ITEM_HIERARCHY')
WMD_STORE_HIER_TABLE = cfg['tables'].get('wmd_store_hierarchy', 'ANALYSIS.FINAL.WMD_SITE_HIERARCHY')

print(f'Markdown log   : {MARKDOWN_LOG_TABLE}')
print(f'Markdown sales : {MARKDOWN_SALES_TABLE}')
print(f'Wastage        : {WASTAGE_TABLE}')
print(f'Item hierarchy : {WMD_ITEM_HIER_TABLE}')
print(f'Store hierarchy: {WMD_STORE_HIER_TABLE}')

# WMD_MARKDOWN_API: SITE_ID→STORE_ID, MARKDOWN_DATE→LOCAL_DATE, MARKDOWN_QUANTITY→QTY_MARKDOWN
markdown_log = (
    session.table(MARKDOWN_LOG_TABLE)
    .with_column('STORE_ID',     F.col('SITE_ID'))
    .with_column('LOCAL_DATE',   F.col('MARKDOWN_DATE'))
    .with_column('QTY_MARKDOWN', F.col('MARKDOWN_QUANTITY'))
    .filter(F.col('LOCAL_DATE') >= WMD_LOOKBACK)
    .filter(F.col('LOCAL_DATE') <= THIS_WEEK_END)
    .cache_result()
)
# WMD_SALES: SITE_ID→STORE_ID, SALE_QUANTITY→MARKDOWN_QUANTITY, SALE_VALUE→MARKDOWN_VALUE
markdown_sales = (
    session.table(MARKDOWN_SALES_TABLE)
    .with_column('STORE_ID',          F.col('SITE_ID'))
    .with_column('MARKDOWN_QUANTITY', F.col('SALE_QUANTITY'))
    .with_column('MARKDOWN_VALUE',    F.col('SALE_VALUE'))
    .filter(F.col('DATE') >= WMD_LOOKBACK)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .cache_result()
)
# WMD_WASTAGE: SITE_ID→STORE_ID, WASTE_DATE→DATE
wastage = (
    session.table(WASTAGE_TABLE)
    .with_column('STORE_ID', F.col('SITE_ID'))
    .with_column('DATE',     F.col('WASTE_DATE'))
    .filter(F.col('DATE') >= WMD_LOOKBACK)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .cache_result()
)

# WMD_ITEM_HIERARCHY: VME_CODE→ITEM_ID, ITEM_NAME→ITEM_DESCRIPTION, HIERARCHY_LEVEL_1_ID→DEPARTMENT
_item_w = Window.partitionBy('VME_CODE').orderBy(F.col('FILE_TIMESTAMP').desc())
wmd_item_hierarchy = (
    session.table(WMD_ITEM_HIER_TABLE)
    .with_column('_RN', F.row_number().over(_item_w))
    .filter(F.col('_RN') == 1)
    .select(
        F.col('VME_CODE').cast(LongType()).alias('ITEM_ID'),
        F.col('ITEM_NAME').alias('ITEM_DESCRIPTION'),
        F.col('HIERARCHY_LEVEL_1_ID').alias('DEPARTMENT'),
        F.col('SECTION'),
        F.col('SUB_SECTION'),
    )
    .filter(F.col('ITEM_ID').is_not_null())
    .cache_result()
)

# WMD_SITE_HIERARCHY: SITE_ID→STORE_ID, SITE_NAME→STORE_NAME
store_hierarchy = (
    session.table(WMD_STORE_HIER_TABLE)
    .with_column('STORE_ID',   F.col('SITE_ID'))
    .with_column('STORE_NAME', F.col('SITE_NAME'))
    .cache_result()
)

print(f'\nMarkdown log rows  : {markdown_log.count():,}')
print(f'Markdown sales rows: {markdown_sales.count():,}')
print(f'Wastage rows       : {wastage.count():,}')
print(f'Item hierarchy rows (deduplicated): {wmd_item_hierarchy.count():,}')
print(f'Store hierarchy rows: {store_hierarchy.count():,}')

Markdown log   : INBOUND.SOURCE.SOURCE_WMD_MARKDOWN_LOG_CONVERTED
Markdown sales : INBOUND.SOURCE.SOURCE_WMD_MARKDOWN_SALES
Wastage        : INBOUND.SOURCE.SOURCE_WMD_WASTAGE
Item hierarchy : INBOUND.SOURCE.SOURCE_WMD_ITEM_HIERARCHY
Store hierarchy: INBOUND.SOURCE.SOURCE_WMD_STORE_HIERARCHY

Markdown log rows  : 138,482
Markdown sales rows: 503,996
Wastage rows       : 53,646
Item hierarchy rows (deduplicated): 179,728
Store hierarchy rows: 97,769


## Stage 1 — Anomaly Detection:  Runs all registered detectors (OSA, Compliance, Markdown, Waste, SKU) then merges results.

In [6]:
cfg

{'connection': {'warehouse': 'SNOWFLAKE_LEARNING_WH',
  'database': 'SNOWFLAKE_LEARNING_DB',
  'schema': 'STORE_INSIGHT',
  'role': 'SNOWFLAKE_LEARNING_ROLE'},
 'tables': {'osa': 'ANALYSIS.FINAL.LSM_ITEM_STORE_DAY_REPORTING_PERIOD',
  'hierarchy': 'ANALYSIS.FINAL.LSM_ITEM_HIERARCHY',
  'feedback': 'ANALYSIS.FINAL.LSM_ALERT_FEEDBACK',
  'markdown_log': 'INBOUND.SOURCE.SOURCE_WMD_MARKDOWN_LOG_CONVERTED',
  'markdown_sales': 'INBOUND.SOURCE.SOURCE_WMD_MARKDOWN_SALES',
  'wastage': 'INBOUND.SOURCE.SOURCE_WMD_WASTAGE',
  'wmd_item_hierarchy': 'INBOUND.SOURCE.SOURCE_WMD_ITEM_HIERARCHY',
  'wmd_store_hierarchy': 'INBOUND.SOURCE.SOURCE_WMD_STORE_HIERARCHY'},
 'output': {'alerts': 'SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.ALERTS_PIPELINE',
  'signals': 'SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.ALERTS_SIGNALS',
  'enriched': 'SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.ALERTS_ENRICHED',
  'run_log': 'SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.ALERTS_RUN_LOG'},
 'detectors': {'enabled': ['osa.wow',
   'osa.vs_chain',
   

In [7]:
from store_insight.pipeline.stages._1_detect.detect import detect_signals
from store_insight.pipeline.pipeline.config import DetectorConfig

det_cfg = DetectorConfig.from_config(cfg)

data = osa.filter(F.col('STORE_ID') == QC_STORE_ID) if QC_STORE_ID else osa

signals = detect_signals(
    data,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    feedback=feedback,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
    wastage=wastage,
    item_hierarchy=wmd_item_hierarchy,
    store_hierarchy=store_hierarchy,
    config=det_cfg,
    cfg=cfg,
)

print(f'Total merged signals: {len(signals)}')

[1_detect] TICK_AND_FLICK=14 (chain avg 32.4%)
[1_detect] MARKDOWN_OVERRIDE VS_CHAIN=10 (chain avg 2.6%)
[1_detect] MARKDOWN_OVERRIDE WOW=10
[1_detect] MARKDOWN_SELLTHROUGH VS_CHAIN=50 (chain avg 135.6%)
[1_detect] MARKDOWN_SELLTHROUGH WOW=51
[1_detect] MARKDOWN_SKU EXCESSIVE_OVERRIDE=72 (lookback=4w, min_rate=50%)
[1_detect] MARKDOWN_SKU REPEAT_REQUEST=2701 (lookback=4w, min_weeks=3, min_requests=8)
[1_detect] WASTE VS_CHAIN=34 (chain avg 28.0%)
[1_detect] WASTE WOW=38
[1_detect] 3189 signals — TICK_AND_FLICK=14, VS_CHAIN=34, WOW=38, EXCESSIVE_OVERRIDE=72, REPEAT_REQUEST=2701, REPEAT=122
Total merged signals: 3189


### Inspect — Signal Breakdown by Type

In [9]:
from collections import Counter

# ── Breakdown by signal_type ──────────────────────────────────────────────────
type_counts = Counter(s['signal_type'] for s in signals)
print('Signals by type:')
for stype, count in sorted(type_counts.items()):
    print(f'  {stype:<25} {count}')

# ── Trigger breakdown within each signal type ─────────────────────────────────
print()
for stype in sorted(type_counts):
    type_sigs = [s for s in signals if s['signal_type'] == stype]
    trigger_counts = Counter(tuple(sorted(s['trigger_types'])) for s in type_sigs)
    print(f'{stype} ({len(type_sigs)} signals):')
    for triggers, count in sorted(trigger_counts.items()):
        print(f'  {"+".join(triggers):<35} {count}')
    print()

Signals by type:
  COMPLIANCE                13
  MARKDOWN_OVERRIDE         13
  MARKDOWN_SELLTHROUGH      57
  MARKDOWN_SKU              18282
  OSA                       574
  WASTE                     37

COMPLIANCE (13 signals):
  TICK_AND_FLICK                      13

MARKDOWN_OVERRIDE (13 signals):
  VS_CHAIN                            7
  VS_CHAIN+WOW                        6

MARKDOWN_SELLTHROUGH (57 signals):
  VS_CHAIN                            52
  VS_CHAIN+WOW                        4
  WOW                                 1

MARKDOWN_SKU (18282 signals):
  EXCESSIVE_OVERRIDE                  141
  EXCESSIVE_OVERRIDE+REPEAT_REQUEST   109
  REPEAT_REQUEST                      18032

OSA (574 signals):
  REPEAT                              228
  REPEAT+VS_CHAIN                     31
  REPEAT+VS_CHAIN+WOW                 37
  REPEAT+WOW                          25
  VS_CHAIN                            91
  VS_CHAIN+WOW                        93
  WOW                         

## Stage 2 — Fact Gathering: Enriches each signal with department breakdowns, drilldowns, day-of-week, chain drift, history, projections.

In [11]:
from store_insight.pipeline.stages._2_facts.facts import enrich_signals

enriched = enrich_signals(
    temp_signals,
    osa,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    projection_weeks=PROJECTION_WEEKS,
    drilldown_top_n=DRILLDOWN_TOP_N,
    feedback=feedback,
    markdown_log=markdown_log,
    markdown_sales=markdown_sales,
    wastage=wastage,
    item_hierarchy=wmd_item_hierarchy,
    cfg=cfg,
)

print(f'{len(enriched)} signals enriched')

[2_facts] 5 signals enriched
5 signals enriched


In [12]:
enriched

[{'signal_type': 'MARKDOWN_SKU',
  'merge_key': ('MARKDOWN_SKU', 242, 202320000009),
  'store_id': 242,
  'site_name': 'Fronks Road',
  'item_id': 202320000009,
  'item_description': '',
  'department': '',
  'trigger_types': ['EXCESSIVE_OVERRIDE'],
  'markdown_request_count': 6,
  'override_count': 3,
  'lookback_weeks': 4,
  'avg_markdown_pct': 50.5,
  'facts': {'sku_markdown_history': [{'week_start': '2025-06-02 00:00:00',
     'request_count': 6,
     'override_count': 3,
     'avg_markdown_pct': 50.5}]}},
 {'signal_type': 'MARKDOWN_SKU',
  'merge_key': ('MARKDOWN_SKU', 633, 5000128104494),
  'store_id': 633,
  'site_name': 'Bishops Stortford',
  'item_id': 5000128104494,
  'item_description': '',
  'department': '',
  'trigger_types': ['EXCESSIVE_OVERRIDE'],
  'markdown_request_count': 6,
  'override_count': 3,
  'lookback_weeks': 4,
  'avg_markdown_pct': 65.33,
  'facts': {'sku_markdown_history': [{'week_start': '2025-05-19 00:00:00',
     'request_count': 6,
     'override_count

### Inspect — Facts QC Checks per Signal Type

In [13]:
from store_insight.pipeline.stages._3_investigate.investigate import investigate

narratives = [investigate(sig, session=session, cfg=cfg) for sig in enriched]

# ── QC: check all four required fields are populated ──────────────────────────
REQUIRED_FIELDS = ['issue', 'explanation', 'cost_if_not_fixed', 'recommended_action']

blank_fields = [
    (enriched[i]['signal_type'], enriched[i]['store_id'], k)
    for i, nar in enumerate(narratives)
    for k in REQUIRED_FIELDS if not nar.get(k)
]

print(f'{len(narratives)} narratives generated')
if blank_fields:
    print(f'WARNING — {len(blank_fields)} blank fields:')
    for b in blank_fields[:10]:
        print(f'  type={b[0]}  store={b[1]}  missing={b[2]}')
else:
    print('All four fields populated for every narrative.')

KeyError: 'this_week_lsv'

In [ ]:
narratives

[{'issue': 'WADGATE ROAD: 59.2% of alerts tick-and-flicked this week',
  'explanation': '45 of 76 alerts worked this week were tick-and-flick (59.2%). This is a +25.4pp increase from last week and +23.8pp above the chain average. Worst alert type by tick-and-flick rate: Availability (59.2%).',
  'cost_if_not_fixed': 'Unresolved alerts mean availability issues go unaddressed, increasing the risk of lost sales across affected departments.',
  'recommended_action': 'Review alert working practice with your team at WADGATE ROAD. Spot-check the Availability alert type and ensure all alerts are genuinely investigated. Consider coaching your team to improve compliance.'},
 {'issue': 'WIMPOLE ROAD: 59.6% of alerts tick-and-flicked this week',
  'explanation': '65 of 109 alerts worked this week were tick-and-flick (59.6%). This is a +16.9pp increase from last week and 24.2pp above the chain average. Worst alert type by tick-and-flick rate: Availability (59.6%).',
  'cost_if_not_fixed': 'Unresolv